# Specimen 04 — Sequential Research Assistant

Goal: combine everything above into one pipeline — the actual final-project codebase, sequential version. This exact code becomes Phase 6's concurrent version later: a plain `for` loop here turns into `asyncio.gather` there, and a plain dict here gets wrapped in an `asyncio.Lock` there. Don't build a second, separate project for Phase 6 — extend this one.

In [1]:
import os
import json
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

def call_model(messages, tools=None, max_tokens=800, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    return client.messages.create(**kwargs)


`call_model` bakes in two things learned the hard way in Phase 3/4: `thinking` is disabled by default, since `claude-opus-5` defaults to extended thinking and can silently burn a small `max_tokens` budget before producing a visible tool call or text block; and `output_schema` makes structured JSON handoffs a one-liner, since Phase 3's chain-of-thought grading bug came from regex-parsing freeform prose instead of forcing a schema. Every agent-to-agent handoff in this phase should go through `output_schema`, not string parsing.

## 1. Planner breaks a broad question into subtopics

Reuse specimen 02's planner, applied to a genuinely broad question (e.g. "compare 4 approaches to X") so it naturally decomposes into independent subtopics.

In [2]:
PLANNER_SYSTEM = (
    "You are a planning agent. Given a broad research question, break it into independent "
    "subtopics that can each be researched separately, without needing another subtopic's findings."
)

PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "subtopics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "key": {"type": "string"},
                    "description": {"type": "string"},
                },
                "required": ["key", "description"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["subtopics"],
    "additionalProperties": False,
}

def run_planner(question, max_subtopics=5):
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{PLANNER_SYSTEM}\n\nBroad question: {question}\n\n"
            f"Break this into at most {max_subtopics} independent subtopics. Each 'key' should be a short slug."
        )}],
        output_schema=PLAN_SCHEMA,
        max_tokens=2000,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    subtopics = json.loads(text)["subtopics"]
    if len(subtopics) > max_subtopics:
        subtopics = subtopics[:max_subtopics]
    return subtopics, response.usage

QUESTION = "Compare four popular Python web frameworks -- Django, Flask, FastAPI, and Tornado -- for building a REST API that needs to serve 10,000 requests per second."
subtopics, planner_usage = run_planner(QUESTION)
for s in subtopics:
    print(f"{s['key']}: {s['description']}")

django-rest-api-performance: Research Django (with Django REST Framework) as a REST API platform: its WSGI/ASGI architecture, sync vs async support, ORM and middleware overhead, published benchmark throughput and latency figures, typical deployment stack (gunicorn/uvicorn workers, Nginx), tuning levers (connection pooling, caching, serializer overhead), and how many instances/cores are realistically needed to approach 10,000 requests per second. Note developer ergonomics, batteries-included features, and ecosystem maturity relevant to API work.
flask-rest-api-performance: Research Flask (with extensions such as Flask-RESTful/Flask-Smorest) as a REST API platform: WSGI foundation, threaded vs gevent vs process-based concurrency, published benchmark throughput and latency figures, blocking I/O implications, deployment options (gunicorn workers, uWSGI, meinheld/gevent), and the horizontal scaling required to reach 10,000 requests per second. Note flexibility, extension ecosystem, and main

## 2. Build the shared store as its own small class

A dict wrapped in a class with `get`/`set` methods, even though nothing needs locking yet (sequential execution means no race conditions). The point is that Phase 6 can drop an `asyncio.Lock` around these same methods without a rewrite.

In [3]:
class ResearchStore:
    def __init__(self):
        self._data = {}

    def set(self, key, value):
        self._data[key] = value

    def get(self, key):
        return self._data.get(key)

    def items(self):
        return list(self._data.items())

    def __len__(self):
        return len(self._data)

store = ResearchStore()
print("Store ready. In Phase 6 this becomes: self._lock = asyncio.Lock(), and set()/get() acquire it -- same shape, no rewrite.")

Store ready. In Phase 6 this becomes: self._lock = asyncio.Lock(), and set()/get() acquire it -- same shape, no rewrite.


## 3. Run one researcher agent per subtopic, one at a time

A plain `for` loop — the sequential half of the eventual concurrency lesson. Each researcher writes its structured result into the shared store built in step 2.

In [4]:
RESEARCHER_SYSTEM = (
    "You are a research agent. Research the given subtopic using your own knowledge and produce a "
    "structured research note. The subtopic may be broad, but the note must stay short regardless: "
    "write a summary of exactly 1-2 sentences, exactly 4 key_facts as short one-line items (pick the "
    "4 most important -- do not try to cover every aspect mentioned in the subtopic), and exactly 2 "
    "open_questions. Brevity matters more than exhaustive coverage."
)

RESEARCH_NOTE_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "key_facts": {"type": "array", "items": {"type": "string"}},
        "open_questions": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "key_facts", "open_questions"],
    "additionalProperties": False,
}

def run_researcher(subtopic_description, feedback=None):
    prompt = f"Subtopic: {subtopic_description}"
    if feedback:
        prompt += f"\n\nA critic reviewed your previous note and said: {feedback}\nProduce a revised note that addresses this."
    response = call_model(
        messages=[{"role": "user", "content": f"{RESEARCHER_SYSTEM}\n\n{prompt}"}],
        output_schema=RESEARCH_NOTE_SCHEMA,
        max_tokens=1200,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return json.loads(text), response.usage

for s in subtopics:
    note, usage = run_researcher(s["description"])
    store.set(s["key"], note)
    print(f"[{s['key']}] researched -- {len(note['key_facts'])} key facts")

print(f"\nStore now holds {len(store)} entries: {[k for k, _ in store.items()]}")

[django-rest-api-performance] researched -- 4 key facts


[flask-rest-api-performance] researched -- 4 key facts


[fastapi-rest-api-performance] researched -- 5 key facts


[tornado-rest-api-performance] researched -- 4 key facts


[benchmark-methodology-and-independent-comparisons] researched -- 4 key facts

Store now holds 5 entries: ['django-rest-api-performance', 'flask-rest-api-performance', 'fastapi-rest-api-performance', 'tornado-rest-api-performance', 'benchmark-methodology-and-independent-comparisons']


## 4. Route each researcher's output through specimen 03's critic

Before it's accepted into the store, capped at 2 revisions, same as specimen 03.

In [5]:
CRITIC_SYSTEM = (
    "You are a critic agent. Review a research note against this rubric. Judge ONLY against the "
    "rubric -- return 'approve' only if every criterion is met, otherwise 'revise' with a specific reason."
)

VERDICT_SCHEMA = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string", "enum": ["approve", "revise"]},
        "reason": {"type": "string"},
    },
    "required": ["verdict", "reason"],
    "additionalProperties": False,
}

RESEARCH_RUBRIC = (
    "1. 'summary' must be non-empty and at least one full sentence.\n"
    "2. 'key_facts' must contain at least 3 distinct, specific facts (not vague restatements of the summary).\n"
    "3. 'open_questions' must contain at least 1 genuine open question."
)

def run_research_critic(note):
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{CRITIC_SYSTEM}\n\nRubric:\n{RESEARCH_RUBRIC}\n\nResearch note (JSON):\n{json.dumps(note)}"
        )}],
        output_schema=VERDICT_SCHEMA,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return json.loads(text), response.usage

def research_with_critic(subtopic, max_revisions=2):
    history = []
    feedback = None
    note, usage = None, None
    total_usages = []
    for attempt_num in range(1, max_revisions + 2):
        note, r_usage = run_researcher(subtopic["description"], feedback)
        total_usages.append(r_usage)
        verdict, c_usage = run_research_critic(note)
        total_usages.append(c_usage)
        history.append({"attempt": attempt_num, "verdict": verdict})
        if verdict["verdict"] == "approve":
            return note, history, total_usages, False
        feedback = verdict["reason"]
    return note, history, total_usages, True

# Re-populate the store, this time gating every entry through the critic before it's accepted.
critic_usages = []
for s in subtopics:
    note, history, usages, capped = research_with_critic(s)
    critic_usages.extend(usages)
    store.set(s["key"], note)
    status = "capped with caveat" if capped else "approved"
    print(f"[{s['key']}] {status} after {len(history)} attempt(s)")
    for h in history:
        print(f"    attempt {h['attempt']}: {h['verdict']['verdict']} -- {h['verdict']['reason']}")

[django-rest-api-performance] approved after 1 attempt(s)
    attempt 1: approve -- Summary is a complete, substantive multi-clause sentence. key_facts contains 4 distinct, specific technical facts (WSGI/ASGI support and sync ORM limits, benchmark comparisons and serializer overhead, concrete production stack components, and specific tuning levers) that go well beyond restating the summary. open_questions contains 2 genuine, non-rhetorical open questions about caching-vs-ORM tradeoffs and Django 5.x async maturity.


[flask-rest-api-performance] approved after 1 attempt(s)
    attempt 1: approve -- Summary is a full, substantive sentence; key_facts contains five distinct, specific technical facts (WSGI blocking model, benchmark figures/overhead, worker-count rules of thumb and server modes, horizontal scaling specifics, ecosystem caveats) rather than restatements; open_questions includes two genuine, non-trivial open questions about gevent vs threaded workers and cost comparison against ASGI alternatives.


[fastapi-rest-api-performance] approved after 1 attempt(s)
    attempt 1: approve -- Summary is a full, substantive multi-clause sentence set. key_facts contains 4 distinct, specific, technically detailed facts (Pydantic v2 Rust core speedup, throughput benchmarks, core/worker scaling config, event-loop blocking pitfalls) that go well beyond restating the summary. open_questions contains 2 genuine, non-rhetorical open research questions about measurement and bottleneck crossover.


[tornado-rest-api-performance] approved after 1 attempt(s)
    attempt 1: approve -- Summary is a full, substantive multi-sentence paragraph. key_facts contains 4 distinct, specific facts with concrete details (history/asyncio integration, concurrency model, benchmark ranges, multi-process scaling strategy) that go beyond restating the summary. open_questions contains 2 genuine, non-rhetorical open questions about unresolved benchmarking and framework choice trade-offs.


[benchmark-methodology-and-independent-comparisons] approved after 1 attempt(s)
    attempt 1: approve -- Summary is a full multi-sentence paragraph; key_facts contains four distinct, specific, technically detailed facts (TechEmpower test types/rankings, throughput vs tail latency and coordinated omission, serializer/payload effects, keep-alive impact) that go beyond the summary; open_questions includes two genuine unresolved research questions.


## 5. A writer agent synthesizes the shared store into one final report

It should only see what's in the store — not the raw researcher agents' intermediate reasoning.

In [6]:
WRITER_SYSTEM = (
    "You are a writer agent. You are given a set of research notes, one per subtopic, all keyed by "
    "subtopic. Synthesize them into one coherent final report that directly answers the original "
    "broad question. Use only what's in these notes -- you have not seen any of the underlying research."
)

def run_writer(question, store):
    notes_blob = json.dumps({key: note for key, note in store.items()}, indent=2)
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{WRITER_SYSTEM}\n\nOriginal question: {question}\n\nResearch notes (JSON):\n{notes_blob}"
        )}],
        max_tokens=700,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return text, response.usage

report, writer_usage = run_writer(QUESTION, store)
print(report)

# Comparing Django, Flask, FastAPI, and Tornado for a 10,000 req/s REST API

## Executive Summary

None of these four frameworks will serve 10,000 requests per second from a single Python process. All four require horizontal scaling across many cores and hosts behind a load balancer; the question is how many cores, how much tuning, and what you give up or gain in developer ergonomics along the way.

The short version:

- **FastAPI** is the strongest default choice for a new, high-throughput JSON API. Its ASGI core (Starlette + uvicorn/uvloop) and Pydantic v2 Rust-based serialization give it the best throughput-per-core of the four, plus typed validation and automatic OpenAPI docs. Realistic target: ~8–16+ modern cores with one uvicorn worker per core.
- **Django + DRF** is the slowest per request but the richest in built-in capability (admin, auth, migrations, ORM, permissions). It can absolutely hit 10k req/s, but expect roughly 8–32+ cores and aggressive caching to get there.
- **Fla

## 6. Wrap the whole run in a lightweight supervisor

Retries a researcher once if it raises an exception, and sums total cost/tokens across every single agent call made during the run (planner + every researcher + every critic pass + writer).

In [7]:
def run_research_assistant(question, max_subtopics=5, max_revisions=2, max_retries=1):
    all_usages = []
    retries_used = 0
    failed_subtopics = []

    subtopics, planner_usage = run_planner(question, max_subtopics=max_subtopics)
    all_usages.append(planner_usage)

    store = ResearchStore()
    for s in subtopics:
        attempt = 0
        while True:
            try:
                note, history, usages, capped = research_with_critic(s, max_revisions=max_revisions)
                all_usages.extend(usages)
                store.set(s["key"], note)
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    failed_subtopics.append({"key": s["key"], "error": str(e)})
                    store.set(s["key"], {"summary": f"Research failed: {e}", "key_facts": [], "open_questions": []})
                    break
                retries_used += 1

    report, writer_usage = run_writer(question, store)
    all_usages.append(writer_usage)

    total_cost = sum(call_cost(u) for u in all_usages)
    summary = {
        "subtopics_researched": len(subtopics),
        "total_api_calls": len(all_usages),
        "retries_used": retries_used,
        "failed_subtopics": failed_subtopics,
        "total_cost": total_cost,
    }
    return report, summary

print("run_research_assistant() defined -- planner -> sequential critic-gated researchers (with retry-on-exception) -> writer, cost-tracked end to end.")

run_research_assistant() defined -- planner -> sequential critic-gated researchers (with retry-on-exception) -> writer, cost-tracked end to end.


## 7. Run it end-to-end on one real broad question

Inspect the final report and the supervisor's retry/cost summary together. This is the artifact Phase 6 will parallelize.

In [8]:
final_report, run_summary = run_research_assistant(QUESTION)

print("FINAL REPORT")
print("=" * 60)
print(final_report)
print()
print("SUPERVISOR SUMMARY")
print("=" * 60)
for k, v in run_summary.items():
    print(f"{k}: {v}")

FINAL REPORT
# Comparing Django, Flask, FastAPI, and Tornado for a 10,000 req/s REST API

## Executive summary

At 10,000 requests per second, the honest answer is that **framework choice is not the dominant variable** — but it does set your per-node efficiency, and therefore your fleet size, cloud bill, and how much engineering effort you spend fighting the framework instead of building features.

None of these four frameworks will serve 10k req/s from a single Python process. CPython's GIL confines one process to roughly one core, so throughput in every case comes from running one worker process per core and scaling horizontally behind a load balancer. The practical differences are:

| Framework | Concurrency model | Typical single-core throughput (plain JSON) | Typical with validation + DB | Path to 10k req/s |
|---|---|---|---|---|
| **FastAPI** | ASGI, native async/await (Starlette + uvicorn/uvloop) | ~5–15k req/s | ~1–3k req/s | Feasible on one well-tuned multi-core node for triv